# **Baseline Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT2",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
student_name = "SUSHRUTA GANGADHAR PATIL"
student_id = "26273312"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

---
## A. Assess Baseline Model

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

### A.1 Generate Predictions with Baseline Model

In [ ]:
# Initialise baseline model - always predicts the majority class (Healthy)
dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)

# Fit on training data
dummy_clf.fit(X_train, y_train.values.ravel())

# Generate predictions on validation and test sets
y_val_pred = dummy_clf.predict(X_val)
y_test_pred = dummy_clf.predict(X_test)

print("Baseline model fitted.")
print(f"Unique predictions (val): {set(y_val_pred)}")
print(f"Unique predictions (test): {set(y_test_pred)}")

### A.2 Selection of Performance Metrics


In [ ]:
macro_f1_val = f1_score(y_val, y_val_pred, average='macro')
print(f"Macro Average F1 (val): {macro_f1_val:.4f}")

print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred))

cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dummy_clf.classes_)
disp.plot(cmap='Blues')
plt.title("Baseline Confusion Matrix (Validation Set)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
performance_metrics_explanations = """

The primary metric chosen is Macro Average F1-score. This metric computes the
F1-score independently for each of the 5 classes and averages them with equal
weight, regardless of class size.

This is appropriate for this dataset because:

1. Class imbalance is severe, the dataset has a 12.5x imbalance ratio between
   Healthy (36.7%) and Scurvy (2.9%). Accuracy is therefore misleading; a model
   that always predicts Healthy achieves ~36.7% accuracy while completely failing
   on minority classes like Scurvy and Night_Blindness.

2. All classes are clinically important, missing a Scurvy or Night_Blindness
   diagnosis has real health consequences. Macro F1 ensures minority classes
   are not ignored during evaluation.

3. Per-class precision, recall and F1 are also reported via classification_report
   to identify which specific diseases the model struggles with.

4. A confusion matrix is included to visualise misclassification patterns,
   particularly how often minority classes are predicted as Healthy.

"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

### A.3 Baseline Model Performance



In [ ]:
baseline_performance_explanations = """

The DummyClassifier (most_frequent strategy) always predicts Healthy, the
majority class for every input regardless of features.

Key observations from the validation set results:

1. All 690 validation samples are predicted as Healthy, 215 Anemia, 173
   Rickets_Osteomalacia, 29 Night_Blindness and 20 Scurvy are all misclassified
   as confirmed by the confusion matrix.

2. Accuracy is 0.37 (37%), this looks deceptively reasonable but is simply the
   proportion of Healthy samples in the validation set. It tells us nothing about
   the model's ability to diagnose disease.

3. Macro Average F1 is 0.1073, Healthy scores an F1 of 0.54 while all other
   4 classes score 0.00. This near-zero macro F1 confirms the model has no
   diagnostic value and sets the performance floor for this project.

4. Minority classes Scurvy (n=20) and Night_Blindness (n=29) score 0.00 across
   precision, recall and F1, the model completely fails to identify these
   clinically important conditions.

5. This baseline establishes that any meaningful model must significantly exceed
   a macro F1 of 0.1073. Subsequent experiments using Decision Tree, Random Forest
   and Gradient Boosting with class_weight='balanced' will target a macro F1
   above 0.80, with particular focus on recall for Scurvy and Night_Blindness.

"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='baseline_performance_explanations', value=baseline_performance_explanations)